# Chapter 8: The Cramér–Rao Lower Bound and Fisher Information

The computational applications of this chapter of *Theory of Statistics*. The code is generated automatically from the book's `code/ch08/` files; see the book for the explanations and proofs.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibrahimguney/statistical-theory/blob/main/notebooks/ch_08.ipynb)

$$
\newcommand{\R}{\mathbb{R}}\newcommand{\N}{\mathbb{N}}\newcommand{\E}{\mathbb{E}}
\newcommand{\P}{\mathbb{P}}\newcommand{\F}{\mathcal{F}}\newcommand{\I}{\mathbb{I}}
\newcommand{\Var}{\operatorname{Var}}\newcommand{\Cov}{\operatorname{Cov}}
\newcommand{\Corr}{\operatorname{Corr}}\newcommand{\MSE}{\operatorname{MSE}}
\newcommand{\Bias}{\operatorname{Bias}}\newcommand{\Normal}{\mathcal{N}}
\newcommand{\iid}{\overset{\text{i.i.d.}}{\sim}}\newcommand{\pto}{\xrightarrow{P}}
\newcommand{\dto}{\xrightarrow{d}}\newcommand{\X}{\mathbf{X}}\newcommand{\x}{\mathbf{x}}
$$

In [1]:
# Setup: numpy, scipy and matplotlib are preinstalled on Colab. Locally:
#   pip install -r code/requirements.txt
# Figure files (PDF) saved by the code are written to the _figures/ folder; the figures are
# also shown inside the notebook.
import os
os.makedirs("_figures", exist_ok=True)
os.chdir("_figures")
%matplotlib inline

## Poisson: $\bar X$ Attains the CRLB

*Source file:* `code/ch08/c08_1_poisson_attains_crlb.py`

In [2]:
import numpy as np

rng = np.random.default_rng(42)
lam = 3.0
for n in [5, 15, 30, 100]:
    samples = rng.poisson(lam=lam, size=(150_000, n))
    var_x = np.var(np.mean(samples, axis=1), ddof=1)
    print(f"n = {n:3d} | CRLB: {lam/n:.6f} | Empirical Var: {var_x:.6f}")

n =   5 | CRLB: 0.600000 | Empirical Var: 0.597129
n =  15 | CRLB: 0.200000 | Empirical Var: 0.199763


n =  30 | CRLB: 0.100000 | Empirical Var: 0.099790


n = 100 | CRLB: 0.030000 | Empirical Var: 0.030091


## Moments of the Score: Checking Lemma 8.6

*Source file:* `code/ch08/c08_2_score_moments.py`

In [3]:
import numpy as np

rng = np.random.default_rng(42)
N = 2_000_000
# Gamma(known shape a, rate b): ln f = a ln b - ln G(a) + (a-1) ln x - b x
a, b = 3.0, 2.0
X = rng.gamma(a, 1/b, size=N)
score = a / b - X              # d/db ln f
hess = -a / b**2 * np.ones(N) # d2/db2 ln f
print(f"Gamma : E[s]={score.mean():+.4f} | Var(s)={score.var():.4f} | -E[s']={-hess.mean():.4f} | theory I={a/b**2:.4f}")

# Cauchy location: s = 2(x-t)/(1+(x-t)^2), I = 1/2
Y = rng.standard_cauchy(N)
s = 2*Y / (1 + Y**2)
h = (2*Y**2 - 2) / (1 + Y**2)**2     # d/dt s
print(f"Cauchy: E[s]={s.mean():+.4f} | Var(s)={s.var():.4f} | -E[s']={-h.mean():.4f} | theory I=0.5000")

# Uniform(0, theta): regularity violated, E[s] != 0
theta = 2.0
print(f"Uniform: E[s] = -1/theta = {-1/theta:.3f}  (not zero!)")

Gamma : E[s]=-0.0009 | Var(s)=0.7509 | -E[s']=0.7500 | theory I=0.7500
Cauchy: E[s]=+0.0007 | Var(s)=0.5001 | -E[s']=0.5005 | theory I=0.5000
Uniform: E[s] = -1/theta = -0.500  (not zero!)


## When the CRLB Is Not Attained and When It Is Violated

*Source file:* `code/ch08/c08_3_crlb_not_attained_or_violated.py`

In [4]:
import numpy as np

rng = np.random.default_rng(42)
R = 300_000
lam = 2.0
print("Exp(lambda): lambda_tilde = (n-1)/sum X")
for n in [5, 20, 100]:
    T = rng.exponential(1/lam, size=(R, n)).sum(axis=1)
    lt = (n - 1) / T
    print(f"  n={n:3d} | mean={lt.mean():.4f} | Var={lt.var():.5f} | theory {lam**2/(n-2):.5f} | CRLB {lam**2/n:.5f}")

theta = 1.0
print("Uniform(0,theta): (n+1)/n * max  vs  'pseudo-CRLB' theta^2/n")
for n in [5, 20, 100]:
    t = (n + 1) / n * rng.uniform(0, theta, size=(R, n)).max(axis=1)
    print(f"  n={n:3d} | Var={t.var():.6f} | theory {theta**2/(n*(n+2)):.6f} | pseudo-CRLB {theta**2/n:.6f}")

Exp(lambda): lambda_tilde = (n-1)/sum X


  n=  5 | mean=2.0012 | Var=1.33392 | theory 1.33333 | CRLB 0.80000


  n= 20 | mean=1.9998 | Var=0.22107 | theory 0.22222 | CRLB 0.20000


  n=100 | mean=2.0003 | Var=0.04089 | theory 0.04082 | CRLB 0.04000
Uniform(0,theta): (n+1)/n * max  vs  'pseudo-CRLB' theta^2/n
  n=  5 | Var=0.028442 | theory 0.028571 | pseudo-CRLB 0.200000
  n= 20 | Var=0.002279 | theory 0.002273 | pseudo-CRLB 0.050000


  n=100 | Var=0.000099 | theory 0.000098 | pseudo-CRLB 0.010000
